In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/movielens-100k-dataset/movielens_100k.csv


# Have a look at the dataset

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel 
import warnings
warnings.filterwarnings('ignore')

In [3]:
data = pd.read_csv("../input/movielens-100k-dataset/movielens_100k.csv")

In [4]:
data.head()

,movie_id,title,year,directors,actors,genres
0,1,toy story,1995,John Lasseter,Tom Hanks Tim Allen Don Rickles Jim Varney Wal...,Animation Adventure Comedy Family Fantasy
1,2,goldeneye,1995,Martin Campbell,Pierce Brosnan Sean Bean Izabella Scorupco Fam...,Action Adventure Thriller
2,3,four rooms,1995,Allison Anders Alexandre Rockwell Robert Rodri...,Sammi Davis Amanda De Cadenet Valeria Golino M...,Comedy
3,4,get shorty,1995,Barry Sonnenfeld,John Travolta Gene Hackman Rene Russo Danny De...,Comedy Crime Thriller
4,5,copycat,1995,Jon Amiel,Sigourney Weaver Holly Hunter Dermot Mulroney ...,Drama Mystery Thriller


In [5]:
data.shape

(1681, 6)

In [6]:
# check number of null entries in each columns of data
data.isnull().sum()

movie_id       0
title          0
year           0
directors    126
actors       126
genres       120
dtype: int64

For content based recommendation we need 'directors','actors','genres' columns.
There for if all the rows are NaN for these three columns we will drop that rows.
Then reset the index so that index range for 0,1,...n-1.

In [7]:
data.dropna(subset=['directors','actors','genres'],axis=0,how='all',inplace=True)
data.reset_index(drop=True,inplace=True)

In [8]:
data.isnull().sum()

movie_id     0
title        0
year         0
directors    9
actors       9
genres       3
dtype: int64

Now we will replace each NaN with empty string ''.

In [9]:
data.fillna('',inplace=True)

In [10]:
data.isnull().sum()

movie_id     0
title        0
year         0
directors    0
actors       0
genres       0
dtype: int64

Now the dataset has no NaN value.

Now we have to drop the duplicate movies from the data.

In [11]:
data.drop_duplicates(subset=['title','year','directors','actors','genres'],
                     ignore_index=True,inplace=True)

In [12]:
data.shape

(1548, 6)

Now we have to create a new column 'combined' which contains the 'director', 'actors', 'genres' of each movie. For this I used a function 'combine_feature'.

In [13]:
import re

def listToString(list):
    str1 = " "
    return (str1.join(list))

def combine_feature(row):
  comb = row['directors']+' '+row['actors']+' '+row['genres']
  lst=re.findall('.[^A-Z]*', comb)
  convertedStr = listToString(lst)
  return convertedStr.replace('  ',' ')

Apply the 'combine_feature' to each row.

Lower each string of combined column.

Remove any punctuation from each string in combined

In [14]:
data['combined'] = data.apply(lambda x: combine_feature(x),axis=1)
data['combined'] = data['combined'].str.lower()
data['combined'] = data['combined'].str.replace('[^\w\s]','')

In [15]:
data.head()

,movie_id,title,year,directors,actors,genres,combined
0,1,toy story,1995,John Lasseter,Tom Hanks Tim Allen Don Rickles Jim Varney Wal...,Animation Adventure Comedy Family Fantasy,john lasseter tom hanks tim allen don rickles ...
1,2,goldeneye,1995,Martin Campbell,Pierce Brosnan Sean Bean Izabella Scorupco Fam...,Action Adventure Thriller,martin campbell pierce brosnan sean bean izabe...
2,3,four rooms,1995,Allison Anders Alexandre Rockwell Robert Rodri...,Sammi Davis Amanda De Cadenet Valeria Golino M...,Comedy,allison anders alexandre rockwell robert rodri...
3,4,get shorty,1995,Barry Sonnenfeld,John Travolta Gene Hackman Rene Russo Danny De...,Comedy Crime Thriller,barry sonnenfeld john travolta gene hackman re...
4,5,copycat,1995,Jon Amiel,Sigourney Weaver Holly Hunter Dermot Mulroney ...,Drama Mystery Thriller,jon amiel sigourney weaver holly hunter dermot...


Now the dataset is ready.

We need title and combined columns.

In [16]:
df = data[['title','combined']]
df.head()

,title,combined
0,toy story,john lasseter tom hanks tim allen don rickles ...
1,goldeneye,martin campbell pierce brosnan sean bean izabe...
2,four rooms,allison anders alexandre rockwell robert rodri...
3,get shorty,barry sonnenfeld john travolta gene hackman re...
4,copycat,jon amiel sigourney weaver holly hunter dermot...


Make object of TfidfVectorizer.

Fit and transform the combined column to the TfidfVectorizer object.

In [17]:
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=0, stop_words='english')
tfidf_matrix = tf.fit_transform(df['combined'])

In [18]:
tfidf_matrix

<1548x318668 sparse matrix of type '<class 'numpy.float64'>'
	with 479926 stored elements in Compressed Sparse Row format>

Making linear kernel score for tfidf_matrix.

In [19]:
linear_similarities = linear_kernel(tfidf_matrix, tfidf_matrix)

In [20]:
linear_similarities.shape

(1548, 1548)

Making series which contains title of movie and their index value.

In [21]:
indices = pd.Series(df.index, index=df.title)
indices.head()

title
toy story     0
goldeneye     1
four rooms    2
get shorty    3
copycat       4
dtype: int64

Now the last step.

Making a function which recommends the top 20 movies.

Output: list of tuple which contains movie title and score

In [22]:
def recommend(movie):
    try:
        idx = indices[movie]
        similar_indices = linear_similarities[idx].argsort()[::-1][:20] 
        similar_items = [(df['title'][i], linear_similarities[idx][i]) for i in similar_indices] 
        return similar_items

    except:
        similar_items = []
        n = len(indices[movie])
        for idx in indices[movie]:
            similar_indices = linear_similarities[idx].argsort()[::-1][:int(20/n)] 
            similar_items.append([(df['title'][i], linear_similarities[idx][i]) for i in similar_indices]) 
        return similar_items

Time to recommend some movie.

In [23]:
data.directors.value_counts()

Alfred Hitchcock      12
                       9
Woody Allen            9
Steven Spielberg       8
Martin Scorsese        7
                      ..
David Carson           1
Jean-Pierre Jeunet     1
Daniel Schmid          1
Alex Zamm              1
Jean Delannoy          1
Name: directors, Length: 1000, dtype: int64

Extract movies directed by Alfred Hitchcock.

Then check the result by recommending one of the movie directed by Alfred Hitchcock. Let's say recommend 'psycho' movie.

In [24]:
data[data.directors=='Alfred Hitchcock']

,movie_id,title,year,directors,actors,genres,combined
178,185,psycho,1960,Alfred Hitchcock,Anthony Perkins Vera Miles John Gavin Janet Le...,Horror Mystery Thriller,alfred hitchcock anthony perkins vera miles jo...
424,443,birds the,1963,Alfred Hitchcock,Rod Taylor Jessica Tandy Suzanne Pleshette Tip...,Drama Horror Mystery Romance,alfred hitchcock rod taylor jessica tandy suza...
459,479,vertigo,1958,Alfred Hitchcock,James Stewart Kim Novak Barbara Bel Geddes Tom...,Mystery Romance Thriller,alfred hitchcock james stewart kim novak barba...
460,480,north by northwest,1959,Alfred Hitchcock,Cary Grant Eva Marie Saint James Mason Jessie ...,Adventure Mystery Thriller,alfred hitchcock cary grant eva marie saint ja...
469,489,notorious,1946,Alfred Hitchcock,Cary Grant Ingrid Bergman Claude Rains Louis C...,Drama Film-Noir Romance Thriller,alfred hitchcock cary grant ingrid bergman cla...
470,490,to catch a thief,1955,Alfred Hitchcock,Cary Grant Grace Kelly Jessie Royce Landis Joh...,Mystery Romance Thriller,alfred hitchcock cary grant grace kelly jessie...
484,505,dial m for murder,1954,Alfred Hitchcock,Ray Milland Grace Kelly Robert Cummings John W...,Crime Thriller,alfred hitchcock ray milland grace kelly rober...
580,603,rear window,1954,Alfred Hitchcock,James Stewart Grace Kelly Wendell Corey Thelma...,Mystery Thriller,alfred hitchcock james stewart grace kelly wen...
584,607,rebecca,1940,Alfred Hitchcock,Laurence Olivier Joan Fontaine George Sanders ...,Drama Mystery Romance Thriller,alfred hitchcock laurence olivier joan fontain...
585,608,spellbound,1945,Alfred Hitchcock,Ingrid Bergman Gregory Peck Michael Chekhov Le...,Film-Noir Mystery Romance Thriller,alfred hitchcock ingrid bergman gregory peck m...


In [25]:
recommend('psycho')

[('psycho', 1.0000000000000002),
 ('39 steps the', 0.051457721633784426),
 ('birds the', 0.04926465929012054),
 ('spellbound', 0.04786234349318737),
 ('rebecca', 0.047609594455156064),
 ('notorious', 0.04593564891519088),
 ('rear window', 0.04447638163534599),
 ('fox and the hound the', 0.043907552090268934),
 ('to catch a thief', 0.036416344255586215),
 ('north by northwest', 0.03600067956516354),
 ('vertigo', 0.03559686670122339),
 ('foreign correspondent', 0.03398411085172678),
 ('around the world in 80 days', 0.03360716317896074),
 ('12 angry men', 0.032620047395664084),
 ('dial m for murder', 0.03026131210117291),
 ('citizen kane', 0.023551728948445348),
 ('herbie rides again', 0.02197676217271078),
 ('burnt offerings', 0.021679521442272368),
 ('fog the', 0.019847777051515844),
 ('its a wonderful life', 0.01964900010087657)]

Our results contain movie directed by Alfred Hitchcock ex:'birds the', 'vertigo', 'north by northwest' etc...

That's it we have implemented a simple movie recommendation system.

Upvote if you like this notebook.